# BrainACTIV: Identifying visuo-semantic properties driving cortical selectivity using diffusion-based image manipulation

**Before running the notebook, please ensure you are using a GPU runtime (T4).**

## Setting up the pipeline

The following cells install the `brainactiv` package, download the required files and configure the pipeline. They should take a few minutes to run on GPU. The installation cell restarts the session when it finishes (so that the newly installed packages are used); once it has reconnected, continue with the next cell.

In [ ]:
# Install the brainactiv package (includes BERG, which provides the brain encoders)
!pip install --quiet git+https://github.com/diegogcerdas/BrainACTIV.git

# Restart the session: Colab has already loaded some of the packages that were just
# upgraded (e.g. Pillow), and mixing old and new versions breaks imports
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

In [ ]:
from huggingface_hub import hf_hub_download

# IP-Adapter weights
ip_adapter_checkpoint = hf_hub_download("h94/IP-Adapter", "models/ip-adapter_sd15.bin")

# Modulation embeddings for subject 1 (already projected onto natural CLIP image embeddings)
!gdown --quiet --folder https://drive.google.com/drive/folders/1_4rNJEhdklBkOt-JeNjrvOdSZaHmF-UE
EMBEDDINGS_DIR = "/content/BrainACTIV_modulation_embeddings"

# Category vectors (Appendix A.1 of our paper)
!gdown --quiet --folder https://drive.google.com/drive/folders/1wnAl2F-y4iaoNO3CtK_ht9QQKMNj2Whi
CATEGORY_VECTORS_DIR = "/content/BrainACTIV_category_vectors"

# Brain encoder weights are downloaded here on first use (~2.2 GB for the DINOv2 encoder)
BERG_DIR = "/content/berg"

In [ ]:
import io
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files
from brainactiv.pipeline import BrainACTIV
from brainactiv.brain_encoders import FWRFEncoder, DINOv2Encoder
from brainactiv.methods.estimators import CategoryEstimator
plt.style.use('ggplot')

SUBJECT = 1

# Initialize the BrainACTIV pipeline (Stable Diffusion, IP-Adapter and CLIP). 
# The modulation embeddings are already projected, so no projection set is needed.
brainactiv = BrainACTIV(projection_set_path=None, ip_adapter_checkpoint=ip_adapter_checkpoint)

# Category presence: cosine similarity between an image's CLIP embedding and each category vector
category_estimator = CategoryEstimator(CATEGORY_VECTORS_DIR)

# Each ROI in this demo, as named by the two brain encoders from BERG:
# fwRF (NSD volume space) and DINOv2 (fsaverage surface)
ROIS = {
    "FFA": (["FFA-1", "FFA-2"], ["FFA-1", "FFA-2"]),
    "EBA": (["EBA"], ["EBA"]),
    "VWFA": (["VWFA-1", "VWFA-2"], ["VWFA-1", "VWFA-2"]),
    "OPA": (["OPA"], ["OPA"]),
    "PPA": (["PPA"], ["PPA"]),
    "RSC": (["RSC"], ["RSC"]),
    "V1": (["V1"], ["V1v", "V1d"]),
    "V2": (["V2"], ["V2v", "V2d"]),
    "V3": (["V3"], ["V3v", "V3d"]),
    "V4": (["hV4"], ["hV4"]),
}
encoders, current_roi = {}, None

## Manipulate an image

In [ ]:
# @title ### Run this cell to load a new reference image.

uploaded = files.upload()
filename = next(iter(uploaded))
image_ref = Image.open(io.BytesIO(uploaded[filename])).convert("RGB").resize(brainactiv.resolution)
image_ref

In [ ]:
roi = 'EBA' # @param ["FFA", "EBA", "VWFA", "OPA", "PPA", "RSC", "V1", "V2", "V3", "V4"]
maximize = True # @param ["False", "True"] {type:"raw"}
alpha = 0.7 # @param {type:"slider", min:0, max:1, step:0.05}
gamma = 0.6 # @param {type:"slider", min:0, max:1, step:0.05}
seed = 42 # @param {type:"integer"}

# Load the brain encoders for this ROI (predictions for subject 1)
if roi != current_roi:
    encoders = {}
    torch.cuda.empty_cache()
    fwrf_rois, dinov2_rois = ROIS[roi]
    encoders = {
        "fwRF": FWRFEncoder(SUBJECT, fwrf_rois, BERG_DIR),
        "DINOv2": DINOv2Encoder(SUBJECT, dinov2_rois, "both", BERG_DIR),
    }
    current_roi = roi

# Load the corresponding modulation embedding and manipulate the image
direction = "max" if maximize else "min"
mod_embed = np.load(f"{EMBEDDINGS_DIR}/subj{SUBJECT}_{roi}_mod_embed_{direction}.npy")
image_new = brainactiv.manipulate(
    image_ref, mod_embed, alpha=alpha, gamma=gamma, seed=seed, do_projection=False,
)

# Predicted activation of the ROI (mean over its voxels / vertices)
activations = {
    name: (encoder.encode(image_ref).mean(), encoder.encode(image_new).mean())
    for name, encoder in encoders.items()
}

# Change in category presence between the reference image and its variation
clip_ref = brainactiv.clip_extractor(image_ref).cpu().numpy()
clip_new = brainactiv.clip_extractor(image_new).cpu().numpy()
category_change = category_estimator.compute(clip_new) - category_estimator.compute(clip_ref)

f, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image_ref)
axes[0].set_title('Reference image')
axes[0].axis('off')
axes[1].imshow(image_new)
axes[1].set_title('Variation')
axes[1].axis('off')
x = np.arange(len(activations))
axes[2].bar(x - 0.2, [ref for ref, _ in activations.values()], width=0.4, label='Reference')
axes[2].bar(x + 0.2, [new for _, new in activations.values()], width=0.4, label='Variation')
axes[2].set_xticks(x, list(activations))
axes[2].set_ylabel('Predicted activation (z-scored)')
axes[2].set_title(f'Predicted activation for {roi} (subject {SUBJECT})')
axes[2].legend()
plt.tight_layout()
plt.show()

order = np.argsort(category_change)
plt.figure(figsize=(8, 5))
plt.barh(
    np.array(category_estimator.categories)[order],
    category_change[order],
    color=['tab:red' if c < 0 else 'tab:blue' for c in category_change[order]],
)
plt.xlabel('Change in cosine similarity to category vector')
plt.title('Change in category presence (variation - reference)')
plt.tight_layout()
plt.show()